# Model III
## Rozpoznawanie instrumentów za pomocą konwolucji (ResNet18)

### Biblioteki

In [1]:
import sys
import os
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

import time

from model import Resnet
from trening import get_device, train_model
from przygotowanie_danych import (
    create_datasets,
    create_loaders
)

Hiperparametry modelu

In [3]:
dataset = "1_1"
model_name = "resnet18"

epochs = 100
patience = 10
batch_size = 32
learning_rate = 0.00001
dropout = 0.3

In [4]:
results_file = Path("wyniki/modele.csv")

if results_file.exists():
    results = pd.read_csv(results_file)

    combination_exists = (
        (results["Dataset"] == dataset) &
        (results["Model"] == model_name) &
        (results["Batch size"] == batch_size) &
        (results["Learning rate"] == learning_rate) &
        (results["Dropout"] == dropout)
    ).any()

    if combination_exists:
        raise ValueError("Ta kombinacja datasetu i hiperparametrów była już testowana. Zatrzymuję program.")

print("Nowa kombinacja. Kontynuuję wykonywanie kodu.")

Nowa kombinacja. Kontynuuję wykonywanie kodu.


In [5]:
existing_numbers = [
    int(f[:-4])
    for f in os.listdir("wyniki")
    if f.endswith(".pth")
]

learning_id = max(existing_numbers, default=0) + 1

weights_filename = f"{learning_id}.pth"

weights_filename

'5.pth'

In [6]:
train_dataset, val_dataset, test_dataset = create_datasets(dataset)

train_loader, val_loader, test_loader = create_loaders(train_dataset, val_dataset, test_dataset, batch_size=batch_size)

In [7]:
device = get_device()
print(device)

model = Resnet(dropout=dropout).to(device)

criterion = nn.MultiLabelSoftMarginLoss()

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

cuda


Pętla treningowa

In [8]:
start_time = time.perf_counter()

model, history, best_epoch, best_val_loss = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=epochs,
    patience=patience,
    model_name= str(weights_filename)
)

training_time = time.perf_counter() - start_time

print(f"Czas treningu: {training_time / 60:.2f} min")

Epoka 1/100 | Train loss: 0.5626 | Val loss: 0.4390
Epoka 2/100 | Train loss: 0.4106 | Val loss: 0.3571
Epoka 3/100 | Train loss: 0.3440 | Val loss: 0.3103
Epoka 4/100 | Train loss: 0.2954 | Val loss: 0.2782
Epoka 5/100 | Train loss: 0.2567 | Val loss: 0.2524
Epoka 6/100 | Train loss: 0.2275 | Val loss: 0.2425
Epoka 7/100 | Train loss: 0.2054 | Val loss: 0.2222
Epoka 8/100 | Train loss: 0.1843 | Val loss: 0.2289
Epoka 9/100 | Train loss: 0.1675 | Val loss: 0.2029
Epoka 10/100 | Train loss: 0.1509 | Val loss: 0.1957
Epoka 11/100 | Train loss: 0.1359 | Val loss: 0.1931
Epoka 12/100 | Train loss: 0.1221 | Val loss: 0.1877
Epoka 13/100 | Train loss: 0.1129 | Val loss: 0.1784
Epoka 14/100 | Train loss: 0.1037 | Val loss: 0.1777
Epoka 15/100 | Train loss: 0.0924 | Val loss: 0.1743
Epoka 16/100 | Train loss: 0.0849 | Val loss: 0.1745
Epoka 17/100 | Train loss: 0.0757 | Val loss: 0.1755
Epoka 18/100 | Train loss: 0.0728 | Val loss: 0.1699
Epoka 19/100 | Train loss: 0.0650 | Val loss: 0.1696
Ep

Zapisanie danych o modelu

In [9]:
experiment = pd.DataFrame([{
    "Learning_ID": learning_id,
    "Dataset": dataset,
    "Model": model_name,
    "Batch size": batch_size,
    "Learning rate": learning_rate,
    "Dropout": dropout,
    "Best epoch": best_epoch,
    "Best val loss": best_val_loss,
    "Training time [min]": training_time / 60
}])

experiment.to_csv(
    results_file,
    mode="a",
    header=not results_file.exists(),
    index=False
)

In [10]:
history_df = pd.DataFrame(history)

history_df.to_csv(
    f"wyniki/history_{learning_id}.csv",
    index_label="epoch"
)